# NN Transfer Learning — Low-MP Baseline → High-MP
Follows the same workflow structure as `model_development.ipynb` (LightGBM reference).
Loads the best Low-MP fold checkpoint from `nn_general_model.ipynb`, fine-tunes on
High-MP data across three freeze scenarios.
Architecture kept identical to baseline: `ImprovedNN` with Kaiming init, RMSELoss, AdamW + ReduceLROnPlateau.

In [ ]:
import copy, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from NN_model import ImprovedNN

## 0. Configuration

In [ ]:
data_prefix   = '../0_data/processed_data/'

label         = 'MP_label'
non_feature_cols = ['SMILES', 'MP', 'Type'] + [label]

# MP threshold — consistent across all models in this project
MP_THRESHOLD = 250   # °C boundary between Low ('L') and High ('H')

# NN training constants — kept identical to general_model_baseline_low_int.ipynb
PATIENCE           = 30
MIN_DELTA          = 0.0
MAX_EPOCHS         = 10**6
SAVE_EVERY_N       = 15
N_FOLDS            = 10
N_TRIALS           = 20
GRAD_CLIP          = 0.5
SCHEDULER_FACTOR   = 0.5
SCHEDULER_PATIENCE = 10

# TL scenarios
SCENARIOS = [
    ('no_freeze',      [0, 0, 0], 0),
    ('freeze_fc1',     [1, 0, 0], 1),
    ('freeze_fc1_fc2', [1, 1, 0], 2),
]

In [ ]:
# ── Identify best Low-MP baseline checkpoint (from nn_general_model.ipynb) ──
BASELINE_MODELS_DIR   = Path('best_models_NN_L')
BASELINE_FOLD_METRICS = BASELINE_MODELS_DIR / 'NN_L_best_models_summary.csv'

fold_metrics  = pd.read_csv(BASELINE_FOLD_METRICS)
best_fold_row = fold_metrics.sort_values('RMSE').iloc[0]
best_fold_idx = int(best_fold_row['Fold'])
BASELINE_CKPT = Path(best_fold_row['Model_Path'])

# Read architecture from checkpoint — ensures strict match when loading weights
ckpt_meta     = torch.load(BASELINE_CKPT, map_location='cpu')
HIDDEN_LAYERS = ckpt_meta.get('hidden_layers', [256, 128, 64])

print(f'Baseline checkpoint : fold {best_fold_idx}')
print(f'  Path              : {BASELINE_CKPT}')
print(f'  Val RMSE          : {best_fold_row["RMSE"]:.4f}')
print(f'  Architecture      : {HIDDEN_LAYERS}')

## 1. Shared Utilities
(Identical to `general_model_baseline_low_int.ipynb` and `nn_general_model.ipynb`)

In [ ]:
class RMSELoss(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.mse = nn.MSELoss()
        self.eps = eps

    def forward(self, y_pred, y_true):
        return torch.sqrt(self.mse(y_pred, y_true) + self.eps)


class EarlyStopper:
    def __init__(self, patience=30, min_delta=0):
        self.patience   = int(patience)
        self.min_delta  = float(min_delta)
        self.counter    = 0
        self.best_loss  = float('inf')
        self.stop       = False
        self.stop_epoch = None

    def early_stop(self, val_loss, epoch=None):
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop       = True
                self.stop_epoch = epoch
        return self.stop


def save_checkpoint(model, optimizer, epoch, train_loss, val_loss,
                    y_train, y_val, val_loader,
                    fold_idx, checkpoint_dir, checkpoint_tracking, is_final=False):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_preds.append(model(xb).cpu().numpy())
    preds_val = np.concatenate(all_preds)

    checkpoint_rmse = float(np.sqrt(mean_squared_error(y_val, preds_val)))
    checkpoint_r2   = r2_score(y_val, preds_val)
    checkpoint_q2   = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)

    checkpoint_filename  = f"checkpoint_epoch_{epoch:03d}{'_final' if is_final else ''}.pt"
    checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename

    torch.save({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss, 'val_loss': val_loss,
        'rmse': checkpoint_rmse, 'r2': checkpoint_r2, 'q2': checkpoint_q2,
        'fold_idx': fold_idx, 'is_final': is_final,
    }, checkpoint_path_full)

    checkpoint_tracking.append({
        'Fold': fold_idx, 'Epoch': epoch,
        'Checkpoint_Filename': checkpoint_filename,
        'Checkpoint_Path': str(checkpoint_path_full),
        'Train_Loss': round(train_loss, 6), 'Val_Loss': round(val_loss, 6),
        'RMSE': round(checkpoint_rmse, 6), 'R2': round(checkpoint_r2, 6),
        'Q2': round(checkpoint_q2, 6), 'Is_Final': is_final,
    })
    print(f"[Fold {fold_idx}] {'Final' if is_final else 'Regular'} checkpoint saved at epoch {epoch} - RMSE: {checkpoint_rmse:.4f}")


def set_freeze_mode(model, freeze_level=0):
    """
    freeze_level:
        0 = train all layers
        1 = freeze first hidden block  (Linear + BatchNorm of block 0)
        2 = freeze first two blocks
        3 = freeze first three blocks  (if present)
    Block layout: [Linear, BatchNorm, ReLU, Dropout] — block_size = 4.
    Only Linear + BatchNorm are frozen (ReLU/Dropout have no learnable params).
    """
    block_size = 4
    for p in model.parameters():
        p.requires_grad = True

    if freeze_level == 0:
        print('Freeze Level 0: all layers trainable')
        return

    n_blocks_total = len(model.network) // block_size
    n_blocks = min(freeze_level, n_blocks_total)
    print(f'Freeze Level {freeze_level}: freezing {n_blocks} block(s) out of {n_blocks_total}')

    for b in range(n_blocks):
        start = b * block_size
        for i in range(start, start + 2):   # Linear [i], BatchNorm [i+1]
            for p in model.network[i].parameters():
                p.requires_grad = False


def plot_training_progress(train_losses, val_losses, early_stop_epoch=None,
                           title='Training and Validation Loss'):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_losses, label='Training Loss')
    plt.plot(epochs, val_losses,   label='Validation Loss')
    if early_stop_epoch is not None:
        plt.axvline(x=early_stop_epoch, color='r', linestyle='--', label='Early Stop')
    else:
        plt.axvline(x=len(train_losses), color='gray', linestyle='--', label='End Epoch')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(title)
    plt.legend(); plt.tight_layout(); plt.show()

## 2. TL Fold Trainer

In [ ]:
def evaluate_fold_TL(
    trial, fold_idx,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    hidden_layers, dropout_rate,
    learning_rate, weight_decay, batch_size,
    freeze_level,
    baseline_ckpt,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA,
    save_checkpoints=False, checkpoint_dir='checkpoints', save_every_n_epochs=SAVE_EVERY_N
):
    """
    Transfer-learning fold trainer using a SINGLE learning rate (no param groups).
    Expects pre-scaled numpy arrays. Identical training loop to baseline.

    Returns: rmse, r2, q2, model, train_losses, val_losses, stop_epoch
    """
    device = torch.device('cpu')
    print(f'Fold {fold_idx}: TL on cpu | freeze={freeze_level} | lr={learning_rate:g}')

    checkpoint_tracking = []
    fold_checkpoint_dir = None
    if save_checkpoints:
        fold_checkpoint_dir = Path(checkpoint_dir) / f'fold_{fold_idx}'
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f'Checkpoints will be saved to: {fold_checkpoint_dir}')

    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train,        dtype=torch.float32, device=device)
    X_val_tensor   = torch.tensor(X_val_scaled,   dtype=torch.float32, device=device)
    y_val_tensor   = torch.tensor(y_val,          dtype=torch.float32, device=device)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor),
                              batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(TensorDataset(X_val_tensor,   y_val_tensor),
                              batch_size=batch_size, shuffle=False, num_workers=0)

    # Build model with SAME architecture as baseline
    model = ImprovedNN(
        input_size   = X_train_scaled.shape[1],
        hidden_layers= hidden_layers,
        dropout_rate = dropout_rate
    ).to(device)

    # Load baseline weights — strict=True enforces architecture match
    state = torch.load(baseline_ckpt, map_location=device)
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'], strict=True)
    else:
        model.load_state_dict(state, strict=True)

    # Apply freeze policy
    set_freeze_mode(model, freeze_level)

    # Optimizer over trainable params only; single LR
    optimizer = optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=learning_rate, weight_decay=weight_decay
    )
    criterion     = RMSELoss()
    scheduler     = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE
    )
    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float('inf')
    best_state    = copy.deepcopy(model.state_dict())
    train_losses, val_losses = [], []
    stop_epoch = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss  = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                preds = model(xb)
                loss  = criterion(preds, yb)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = copy.deepcopy(model.state_dict())

        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            save_checkpoint(model, optimizer, epoch, train_loss, val_loss,
                            y_train, y_val, val_loader,
                            fold_idx, fold_checkpoint_dir, checkpoint_tracking)

        if early_stopper.early_stop(val_loss, epoch=epoch):
            stop_epoch = early_stopper.stop_epoch
            print(f'[Fold {fold_idx}] Early stopping at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})')
            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(model, optimizer, epoch, train_loss, val_loss,
                                y_train, y_val, val_loader,
                                fold_idx, fold_checkpoint_dir, checkpoint_tracking, is_final=True)
            break

        if epoch % 50 == 0 or epoch == 1:
            print(f'[Fold {fold_idx}] Epoch {epoch:4d} | Train {train_loss:.4f} | Val {val_loss:.4f} | ES {early_stopper.counter}/{patience}')

    model.load_state_dict(best_state)
    model.eval()

    if save_checkpoints and checkpoint_tracking:
        df_ckpt = pd.DataFrame(checkpoint_tracking)
        df_ckpt.to_csv(fold_checkpoint_dir / f'fold_{fold_idx}_checkpoints.csv', index=False)
        print(f'[Fold {fold_idx}] Checkpoint spreadsheet saved')
        print(f'[Fold {fold_idx}] Total checkpoints saved: {len(checkpoint_tracking)}')

    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_preds.append(model(xb).cpu().numpy())
    preds_val = np.concatenate(all_preds)

    rmse = float(np.sqrt(mean_squared_error(y_val, preds_val)))
    r2   = r2_score(y_val, preds_val)
    q2   = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)

    return rmse, r2, q2, model, train_losses, val_losses, stop_epoch

## 3. Load High-MP Data & Build Folds

In [ ]:
# High-MP scaled data — use the Low-MP feature set so input_size matches baseline
df_high_scaled = pd.read_parquet(
    data_prefix + 'data_with_selected_features_NN_L_scaled.parquet'
)

df_train_H = df_high_scaled[
    (df_high_scaled['Type'] == 'Train') & (df_high_scaled[label] == 'H')
].reset_index(drop=True)

feature_cols = [c for c in df_train_H.columns if c not in non_feature_cols]

X = df_train_H[feature_cols].to_numpy(np.float32)
y = df_train_H['MP'].to_numpy(np.float32)

# Stratify on MP_label (all 'H' here, so bin by quartile as fallback)
y_strat = pd.qcut(pd.Series(y), q=10, labels=False, duplicates='drop').astype(str).to_numpy()

skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(X, y_strat))

print(f'High-MP train samples : {len(X)}')
print(f'Features (Low-MP set) : {len(feature_cols)}')
print(f'Baseline checkpoint   : {BASELINE_CKPT}')
print(f'Hidden layers         : {HIDDEN_LAYERS}')

## 4. Model Development — HP Search per Scenario (mirrors LightGBM trial structure)

In [ ]:
def model_development_tl(tag, freeze_level, trials=N_TRIALS):
    """
    Mirrors model_development() structure:
      - Trial 0: default HPs
      - Trials 1-N: random search
    Returns trial_results dict and best_params.
    """
    rng = np.random.default_rng(42)

    def run_cv(dropout_rate, learning_rate, batch_size, weight_decay):
        fold_rmses = []
        for fold_idx, (tr_idx, va_idx) in enumerate(folds):
            rmse, _, _, _, _, _, _ = evaluate_fold_TL(
                trial=None, fold_idx=fold_idx,
                X_train_scaled=X[tr_idx], y_train=y[tr_idx],
                X_val_scaled=X[va_idx],   y_val=y[va_idx],
                hidden_layers=HIDDEN_LAYERS,
                dropout_rate=dropout_rate,
                learning_rate=learning_rate, weight_decay=weight_decay,
                batch_size=batch_size, freeze_level=freeze_level,
                baseline_ckpt=BASELINE_CKPT,
                max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA,
                save_checkpoints=False
            )
            fold_rmses.append(rmse)
        return fold_rmses

    trial_results = {}

    # Trial 0: default
    DEFAULT_LR      = 1e-3
    DEFAULT_BATCH   = 32
    DEFAULT_DROPOUT = 0.2
    DEFAULT_WD      = 1e-4

    print(f'Trial  0 (default) | running...')
    fold_rmses_0 = run_cv(DEFAULT_DROPOUT, DEFAULT_LR, DEFAULT_BATCH, DEFAULT_WD)
    mean_0 = float(np.mean(fold_rmses_0))
    std_0  = float(np.std(fold_rmses_0))
    trial_results[0] = {
        'fold_rmses': fold_rmses_0, 'mean_rmse': mean_0, 'std_rmse': std_0,
        'params': {'dropout_rate': DEFAULT_DROPOUT, 'learning_rate': DEFAULT_LR,
                   'batch_size': DEFAULT_BATCH, 'weight_decay': DEFAULT_WD}
    }
    print(f'Trial  0 (default) | mean RMSE: {mean_0:.4f} ± {std_0:.4f}')

    # Trials 1-N: random search
    for i in range(1, trials + 1):
        hp = {
            'dropout_rate' : float(rng.uniform(0.2, 0.5)),
            'learning_rate': float(np.exp(rng.uniform(np.log(1e-5), np.log(1e-3)))),
            'weight_decay' : float(np.exp(rng.uniform(np.log(1e-6), np.log(1e-3)))),
            'batch_size'   : int(rng.choice([16, 32, 64])),
        }
        print(f'Trial {i:>2d} | running... {hp}')
        fold_rmses = run_cv(hp['dropout_rate'], hp['learning_rate'],
                            hp['batch_size'], hp['weight_decay'])
        mean_rmse  = float(np.mean(fold_rmses))
        std_rmse   = float(np.std(fold_rmses))
        trial_results[i] = {'fold_rmses': fold_rmses, 'mean_rmse': mean_rmse,
                             'std_rmse': std_rmse, 'params': hp}
        print(f'Trial {i:>2d} | mean RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}')

    best_trial_idx = min(trial_results, key=lambda t: trial_results[t]['mean_rmse'])
    best_params    = trial_results[best_trial_idx]['params']
    print(f'\nBest trial: {best_trial_idx} | mean RMSE: {trial_results[best_trial_idx]["mean_rmse"]:.4f}')
    print(f'Best params: {best_params}')

    return trial_results, best_params

In [ ]:
tl_trial_results = {}
tl_best_params   = {}

for tag, freeze_vec, freeze_level in SCENARIOS:
    print(f'\n=== Scenario: {tag}  |  freeze={freeze_vec}  (level={freeze_level}) ===')

    trial_results, best_params = model_development_tl(tag, freeze_level, trials=N_TRIALS)
    tl_trial_results[tag] = trial_results
    tl_best_params[tag]   = best_params

    with open(f'model_development_results_TL_{tag}.pkl', 'wb') as f:
        pickle.dump(trial_results, f)

## 5. Performance Plot (per scenario)

In [ ]:
def plot_model_performance(model_development_results_dict, title_suffix=''):
    trials     = sorted(model_development_results_dict.keys())
    mean_rmses = np.array([model_development_results_dict[t]['mean_rmse'] for t in trials])
    std_rmses  = np.array([model_development_results_dict[t]['std_rmse']  for t in trials])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(trials, mean_rmses, marker='o', linewidth=1.5, color='steelblue', label='Mean RMSE')
    ax.fill_between(trials,
                    mean_rmses - std_rmses,
                    mean_rmses + std_rmses,
                    alpha=0.25, color='steelblue', label='± 1 std')
    ax.axvline(x=0, color='grey', linestyle='--', linewidth=1, label='Default HP (trial 0)')

    best_trial = trials[int(np.argmin(mean_rmses))]
    best_rmse  = float(min(mean_rmses))
    ax.scatter([best_trial], [best_rmse], color='red', zorder=5,
               label=f'Best (trial {best_trial}, RMSE={best_rmse:.4f})')

    ax.set_xlabel('HP Tuning Iteration', fontsize=12)
    ax.set_ylabel('RMSE', fontsize=12)
    ax.set_title(f'Model Performance vs. HP Tuning Iteration{title_suffix}', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
for tag, _, _ in SCENARIOS:
    print(f'\n=== Performance plot: TL {tag} ===')

    with open(f'model_development_results_TL_{tag}.pkl', 'rb') as f:
        model_development_results = pickle.load(f)

    plot_model_performance(model_development_results, title_suffix=f' — TL {tag}')

## 6. Final Per-fold Training with Best Hyperparameters

In [ ]:
for tag, freeze_vec, freeze_level in SCENARIOS:

    print(f'\n=== Final fold training | scenario={tag} ===')

    best = tl_best_params[tag]

    final_dir = Path(f'best_models_TL_{tag}')
    final_dir.mkdir(parents=True, exist_ok=True)

    final_fold_metrics = []

    for fold_idx, (tr_idx, va_idx) in enumerate(folds):
        rmse, r2, q2, model, train_losses, val_losses, stop_epoch = evaluate_fold_TL(
            trial=None, fold_idx=fold_idx,
            X_train_scaled=X[tr_idx], y_train=y[tr_idx],
            X_val_scaled=X[va_idx],   y_val=y[va_idx],
            hidden_layers=HIDDEN_LAYERS,
            dropout_rate=best['dropout_rate'],
            learning_rate=best['learning_rate'],
            weight_decay=best['weight_decay'],
            batch_size=best['batch_size'],
            freeze_level=freeze_level,
            baseline_ckpt=BASELINE_CKPT,
            max_epochs=MAX_EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA,
            save_checkpoints=True,
            checkpoint_dir=str(final_dir / f'checkpoints_fold_{fold_idx}'),
            save_every_n_epochs=SAVE_EVERY_N
        )

        model_path = final_dir / f'TL_{tag}_best_fold_{fold_idx}.pt'
        torch.save({
            'model_state_dict': model.state_dict(),
            'hidden_layers': HIDDEN_LAYERS,
            'dropout_rate': best['dropout_rate'],
            'learning_rate': best['learning_rate'],
            'weight_decay': best['weight_decay'],
            'batch_size': best['batch_size'],
            'freeze_level': freeze_level,
            'fold_idx': fold_idx,
            'rmse': rmse, 'r2': r2, 'q2': q2,
        }, model_path)
        print(f'[Fold {fold_idx}] Saved best model to: {model_path}')

        final_fold_metrics.append({
            'Fold': fold_idx, 'RMSE': rmse, 'R2': r2, 'Q2': q2,
            'Stop_Epoch': stop_epoch, 'Model_Path': str(model_path),
        })

    metrics_df   = pd.DataFrame(final_fold_metrics)
    metrics_path = final_dir / f'TL_{tag}_best_models_summary.csv'
    metrics_df.to_csv(metrics_path, index=False)
    print(f'\n✅ Saved summary to: {metrics_path}')
    print(metrics_df)

## 7. Test Evaluation — Overall + Low-MP + High-MP (per scenario)

In [ ]:
# Test set uses the same Low-MP feature set (scaled), so input_size is consistent
df_test_scaled = pd.read_parquet(
    data_prefix + 'data_with_selected_features_NN_L_scaled.parquet'
)
df_test      = df_test_scaled[df_test_scaled['Type'] == 'Test']
feature_cols = [c for c in df_test.columns if c not in non_feature_cols]

X_test = torch.tensor(df_test[feature_cols].to_numpy(np.float32), dtype=torch.float32)
y_true = df_test['MP'].to_numpy(float)

all_test_metrics = []

for tag, _, _ in SCENARIOS:

    print(f'\n=== Test evaluation: {tag} ===')

    final_dir   = Path(f'best_models_TL_{tag}')
    metrics_df  = pd.read_csv(final_dir / f'TL_{tag}_best_models_summary.csv')
    best_row    = metrics_df.sort_values('RMSE').iloc[0]
    best_fold   = int(best_row['Fold'])
    print(f'  Using fold {best_fold} (Val RMSE={best_row["RMSE"]:.4f})')

    ckpt         = torch.load(best_row['Model_Path'], map_location='cpu')
    hidden_layers = ckpt['hidden_layers']
    dropout_rate  = ckpt['dropout_rate']

    model = ImprovedNN(
        input_size=len(feature_cols),
        hidden_layers=hidden_layers,
        dropout_rate=dropout_rate
    )
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'], strict=True)
    else:
        model.load_state_dict(ckpt, strict=True)
    model.eval()

    with torch.no_grad():
        y_pred = model(X_test).cpu().numpy().astype(float)

    out_df = pd.DataFrame({
        'SMILES'            : df_test['SMILES'].values,
        'MP_category_default': df_test[label].values,
        'exp MP'            : y_true,
        'pred MP'           : y_pred,
        'error'             : y_pred - y_true,
        'abs_error'         : np.abs(y_pred - y_true),
    })

    def _m(sub):
        if len(sub) == 0:
            return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan, 'n': 0}
        return {
            'RMSE': float(np.sqrt(mean_squared_error(sub['exp MP'], sub['pred MP']))),
            'MAE' : float(mean_absolute_error(sub['exp MP'], sub['pred MP'])),
            'R2'  : float(r2_score(sub['exp MP'], sub['pred MP'])),
            'n'   : len(sub),
        }

    cat = out_df['MP_category_default'].str.strip().str.upper()
    m_all, m_L, m_H = _m(out_df), _m(out_df[cat == 'L']), _m(out_df[cat == 'H'])

    rmse_total = float(np.sqrt(np.mean(out_df['error']**2)))
    print(f'Total RMSE (all): {rmse_total:.3f}')
    print(f'RMSE (Low-MP):    {m_L["RMSE"]:.3f}  (n={m_L["n"]})')
    print(f'RMSE (High-MP):   {m_H["RMSE"]:.3f}  (n={m_H["n"]})')

    out_df.to_csv(f'test_TL_MP_predictions_{tag}.csv', index=False)
    print(f'Saved predictions → test_TL_MP_predictions_{tag}.csv')

    all_test_metrics.append({
        'scenario': tag,
        **{f'overall_{k}': v for k, v in m_all.items()},
        **{f'low_{k}': v    for k, v in m_L.items()},
        **{f'high_{k}': v   for k, v in m_H.items()},
    })

pd.DataFrame(all_test_metrics).to_csv('tl_test_metrics_summary.csv', index=False)
print('\n✅ TL evaluation complete.')
print(pd.DataFrame(all_test_metrics).to_string(index=False))